### Kofam Filtering
This notebook has the purpose to take the output from KofamScan and retrieve the best K00000 hits for each CDS. For instance, the BGC table used here already contains the following informations: antiSMASH, BIG-SCAPE, POEM (operon info) and UniProt alignment. Here we add the pathways information for each BGC. For input in this notebook we are using the output from kofamscan.

### 1. Filtering and preparing the data:

In [36]:
import pandas as pd
import requests
import re

In [37]:
data = []

with open('./test.out') as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        
        parts = line.strip().split(maxsplit=6)
        
        if parts[0] == '*':
            flag = 1
            parts = parts[1:]  
        else:
            flag = 0
        
        while len(parts) < 6:
            parts.append(None)
        
        data.append([flag] + parts)

import pandas as pd

df = pd.DataFrame(
    data,
    columns=['best_hits','query', 'KO', 'threshold', 'score', 'evalue', 'description', 'function']
)
df

,best_hits,query,KO,threshold,score,evalue,description,function
0,0,MGYG000296020_ctg64_1,K21401,233.13,99.2,2.1e-29,menaquinone-9,beta-reductase [EC:1.3.99.38]
1,0,MGYG000296020_ctg64_1,K17830,243.17,75.1,4.8e-22,digeranylgeranylglycerophospholipid,reductase [EC:1.3.1.101 1.3.7.11]
2,0,MGYG000296020_ctg64_1,K03185,422.00,51.0,7.8e-15,2-octaprenyl-6-methoxyphenol,hydroxylase [EC:1.14.13.-]
3,0,MGYG000296020_ctg64_1,K03184,484.83,45.6,3.5e-13,3-demethoxyubiquinol,3-hydroxylase [EC:1.14.99.60]
4,0,MGYG000296020_ctg64_1,K14257,270.90,44.9,5.8e-13,tetracycline,7-halogenase / FADH2 O2-dependent halogenase [...
...,...,...,...,...,...,...,...,...
57573,0,MGYG000296015_ctg9_28,K21785,279.63,17.8,0.00012,"4-hydroxy-2,2'-bipyrrole-5-methanol",dehydrogenase
57574,0,MGYG000296015_ctg9_28,K10679,166.67,11.7,0.012,nitroreductase,/ dihydropteridine reductase [EC:1.-.-.- 1.5.1...
57575,0,MGYG000296015_ctg9_28,K03577,235.53,11.4,0.018,TetR/AcrR,"family transcriptional regulator, acrAB operon..."
57576,0,MGYG000296015_ctg9_29,K12189,52.17,11.6,0.012,ESCRT-II,complex subunit VPS25


In [38]:
df["evalue"] = pd.to_numeric(df["evalue"], errors="coerce")
df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["threshold"] = pd.to_numeric(df["threshold"], errors="coerce")
df.dtypes

best_hits        int64
query              str
KO                 str
threshold      float64
score          float64
evalue         float64
description        str
function           str
dtype: object

In [39]:
df = df[df["evalue"] <= 1.0e-5]
df

,best_hits,query,KO,threshold,score,evalue,description,function
0,0,MGYG000296020_ctg64_1,K21401,233.13,99.2,2.100000e-29,menaquinone-9,beta-reductase [EC:1.3.99.38]
1,0,MGYG000296020_ctg64_1,K17830,243.17,75.1,4.800000e-22,digeranylgeranylglycerophospholipid,reductase [EC:1.3.1.101 1.3.7.11]
2,0,MGYG000296020_ctg64_1,K03185,422.00,51.0,7.800000e-15,2-octaprenyl-6-methoxyphenol,hydroxylase [EC:1.14.13.-]
3,0,MGYG000296020_ctg64_1,K03184,484.83,45.6,3.500000e-13,3-demethoxyubiquinol,3-hydroxylase [EC:1.14.99.60]
4,0,MGYG000296020_ctg64_1,K14257,270.90,44.9,5.800000e-13,tetracycline,7-halogenase / FADH2 O2-dependent halogenase [...
...,...,...,...,...,...,...,...,...
57566,0,MGYG000296015_ctg9_28,K19285,300.13,41.5,9.400000e-12,FMN,reductase (NADPH) [EC:1.5.1.38]
57567,0,MGYG000296015_ctg9_28,K24130,347.10,39.1,4.800000e-11,glutaredoxin-dependent,peroxiredoxin [EC:1.11.1.25]
57568,0,MGYG000296015_ctg9_28,K15057,423.70,34.1,1.500000e-09,nitrobenzene,nitroreductase [EC:1.7.1.16]
57569,0,MGYG000296015_ctg9_28,K09019,115.50,29.4,5.500000e-08,3-hydroxypropanoate,dehydrogenase [EC:1.1.1.-]


In [40]:
df["query"].nunique()

1334

In [41]:
df_best_hits = df[df["best_hits"] == 1]
df_best_hits.drop(columns={"best_hits", "threshold", "score", "function"}, inplace=True)
df_best_hits

,query,KO,evalue,description
280,MGYG000296020_ctg64_7,K04077,5.200000e-276,chaperonin GroEL [EC:5.6.1.7]
297,MGYG000296020_ctg64_8,K04078,3.300000e-40,chaperonin GroES
299,MGYG000296020_ctg64_9,K03075,7.200000e-24,preprotein translocase subunit SecG
352,MGYG000296020_ctg64_13,K06168,8.900000e-196,tRNA-2-methylthio-N6-dimethylallyladenosine sy...
393,MGYG000296020_ctg1_26,K03723,0.000000e+00,transcription-repair coupling factor (superfam...
...,...,...,...,...
57526,MGYG000296015_ctg9_23,K06192,7.800000e-157,intermembrane transport protein PqiB/LetB
57528,MGYG000296015_ctg9_24,K09857,1.900000e-43,intermembrane transport lipoprotein PqiC
57531,MGYG000296015_ctg9_25,K00570,9.700000e-34,phosphatidylethanolamine/phosphatidyl-N-methyl...
57558,MGYG000296015_ctg9_26,K09978,9.800000e-30,uncharacterized protein


In [42]:
df_without_best_hits = df[~df["query"].isin(df_best_hits["query"])]
df_without_best_hits["query"].nunique()

469

In [43]:
df_without_best_hits = (
    df_without_best_hits
    .sort_values(["best_hits", "evalue"], ascending=[False, True])
    .drop_duplicates("query")
)
df_without_best_hits.drop(columns={"best_hits", "threshold", "score", "description"}, inplace=True)
df_without_best_hits.rename(columns={"function": "description", "('query', None)": "query"}, inplace=True)
df_without_best_hits

,query,KO,evalue,description
15590,MGYG000296065_ctg187_2,K16410,0.000000,polyketide synthase StiF
16057,MGYG000296065_ctg23_3,K16124,0.000000,synthetase III
16343,MGYG000296065_ctg23_4,K28602,0.000000,gramicidin synthase LgrC
16696,MGYG000296065_ctg23_6,K16124,0.000000,synthetase III
17816,MGYG000296065_ctg72_16,K16124,0.000000,synthetase III
...,...,...,...,...
46230,MGYG000296076_ctg4_75,K07762,0.000007,[EC:3.4.24.79]
19620,MGYG000296065_ctg82_7,K13663,0.000008,[EC:2.3.1.-]
23410,MGYG000296065_ctg60_8,K00694,0.000009,synthase (UDP-forming) [EC:2.4.1.12]
56044,MGYG000296015_ctg33_8,K02742,0.000009,protein


In [44]:
df_best_hits["quality"] = 1
df_without_best_hits["quality"] = 0
final_df = pd.concat([df_best_hits, df_without_best_hits], ignore_index=True)
final_df = final_df[~final_df["query"].isna() & (final_df["query"] != "NaN")]
final_df

,query,KO,evalue,description,quality
0,MGYG000296020_ctg64_7,K04077,5.200000e-276,chaperonin GroEL [EC:5.6.1.7],1
1,MGYG000296020_ctg64_8,K04078,3.300000e-40,chaperonin GroES,1
2,MGYG000296020_ctg64_9,K03075,7.200000e-24,preprotein translocase subunit SecG,1
3,MGYG000296020_ctg64_13,K06168,8.900000e-196,tRNA-2-methylthio-N6-dimethylallyladenosine sy...,1
4,MGYG000296020_ctg1_26,K03723,0.000000e+00,transcription-repair coupling factor (superfam...,1
...,...,...,...,...,...
1372,MGYG000296076_ctg4_75,K07762,7.000000e-06,[EC:3.4.24.79],0
1373,MGYG000296065_ctg82_7,K13663,8.100000e-06,[EC:2.3.1.-],0
1374,MGYG000296065_ctg60_8,K00694,8.900000e-06,synthase (UDP-forming) [EC:2.4.1.12],0
1375,MGYG000296015_ctg33_8,K02742,9.000000e-06,protein,0


In [45]:
final_df["BGC_ID"] = final_df["query"].str.extract(r"(MGYG\d+).*?ctg(\d+)") \
    .agg("_".join, axis=1)
final_df["gene_id"] = final_df["query"].apply(
    lambda x: "_".join(x.split("_")[1:]) if isinstance(x, str) else None
)

final_df["percentual"] = final_df.groupby("BGC_ID")["quality"].transform("sum") / final_df.groupby("BGC_ID")["BGC_ID"].transform("size") * 100

final_df_filtered = final_df.drop(columns=["query", "evalue", "description"])
final_df_filtered

,KO,quality,BGC_ID,gene_id,percentual
0,K04077,1,MGYG000296020_64,ctg64_7,44.444444
1,K04078,1,MGYG000296020_64,ctg64_8,44.444444
2,K03075,1,MGYG000296020_64,ctg64_9,44.444444
3,K06168,1,MGYG000296020_64,ctg64_13,44.444444
4,K03723,1,MGYG000296020_1,ctg1_26,69.230769
...,...,...,...,...,...
1372,K07762,0,MGYG000296076_4,ctg4_75,54.838710
1373,K13663,0,MGYG000296065_82,ctg82_7,60.000000
1374,K00694,0,MGYG000296065_60,ctg60_8,50.000000
1375,K02742,0,MGYG000296015_33,ctg33_8,76.923077


### 2. Using KEGG API to obtain pathways information

In [46]:
url = "https://rest.kegg.jp/link/pathway/ko"
r = requests.get(url)

lines = r.text.strip().split("\n")

data = [line.split("\t") for line in lines if "\t" in line]

df_kegg = pd.DataFrame(data, columns=["KO", "pathway"])

df_kegg["KO"] = df_kegg["KO"].str.replace("ko:", "")
df_kegg["pathway"] = df_kegg["pathway"].str.replace("path:", "")
df_kegg

,KO,pathway
0,K00001,map00010
1,K00001,ko00010
2,K00002,map00010
3,K00002,ko00010
4,K00016,map00010
...,...,...
97035,K20607,ko05418
97036,K21278,map05418
97037,K21278,ko05418
97038,K23790,map05418


In [47]:
url = "https://rest.kegg.jp/list/pathway"
r = requests.get(url)

lines = r.text.strip().split("\n")

data = [line.split("\t") for line in lines if "\t" in line]

df_list = pd.DataFrame(data, columns=["pathway", "description"])
df_list

,pathway,description
0,map01100,Metabolic pathways
1,map01110,Biosynthesis of secondary metabolites
2,map01120,Microbial metabolism in diverse environments
3,map01200,Carbon metabolism
4,map01210,2-Oxocarboxylic acid metabolism
...,...,...
580,map07035,Prostaglandins
581,map07110,Benzoic acid family
582,map07112,"1,2-Diphenyl substitution family"
583,map07114,Naphthalene family


In [48]:
df_merge = final_df_filtered.merge(df_kegg, on="KO", how="left")
df_merge = df_merge.loc[df_merge["pathway"].str.startswith("map")]
df_merge = df_merge.merge(df_list, on="pathway", how="left")
df_merge

,KO,quality,BGC_ID,gene_id,percentual,pathway,description
0,K04077,1,MGYG000296020_64,ctg64_7,44.444444,map03018,RNA degradation
1,K04077,1,MGYG000296020_64,ctg64_7,44.444444,map04212,Longevity regulating pathway - worm
2,K04077,1,MGYG000296020_64,ctg64_7,44.444444,map04940,Type I diabetes mellitus
3,K04077,1,MGYG000296020_64,ctg64_7,44.444444,map05134,Legionellosis
4,K04077,1,MGYG000296020_64,ctg64_7,44.444444,map05152,Tuberculosis
...,...,...,...,...,...,...,...
2520,K10302,0,MGYG000296059_35,ctg35_10,54.545455,map05132,Salmonella infection
2521,K13663,0,MGYG000296065_82,ctg82_7,60.000000,map00543,Exopolysaccharide biosynthesis
2522,K00694,0,MGYG000296065_60,ctg60_8,50.000000,map00500,Starch and sucrose metabolism
2523,K00694,0,MGYG000296065_60,ctg60_8,50.000000,map01100,Metabolic pathways


In [49]:
df_groupby = df_merge.groupby("BGC_ID").agg({
    "pathway": lambda x: ";".join(x.dropna().unique()),
    "description": lambda x: ";".join(x.dropna().unique()),
    "percentual": "first"
}).reset_index()
df_groupby

,BGC_ID,pathway,description,percentual
0,MGYG000296006_405,map00790;map01100;map01240;map00900;map01110,Folate biosynthesis;Metabolic pathways;Biosynt...,50.000000
1,MGYG000296008_2,map00400;map01100;map01110;map01230;map02024;m...,"Phenylalanine, tyrosine and tryptophan biosynt...",76.470588
2,MGYG000296008_22,map00984;map01100;map01120;map00360;map00362;m...,Steroid degradation;Metabolic pathways;Microbi...,33.333333
3,MGYG000296008_6,map03010;map03020;map03420;map03030;map03430;m...,Ribosome;RNA polymerase;Nucleotide excision re...,83.333333
4,MGYG000296009_1,map00860;map00970;map01100;map01110;map01120;m...,Porphyrin metabolism;Aminoacyl-tRNA biosynthes...,85.000000
...,...,...,...,...
94,MGYG000296075_3,map00230;map01100;map01232;map00600;map00511;m...,Purine metabolism;Metabolic pathways;Nucleotid...,30.769231
95,MGYG000296075_4,map00970;map02020;map02040;map05111;map00300;m...,Aminoacyl-tRNA biosynthesis;Two-component syst...,27.272727
96,MGYG000296076_1,map00260;map01100;map01110;map01120;map01230;m...,"Glycine, serine and threonine metabolism;Metab...",71.428571
97,MGYG000296076_10,map02024;map04075;map00906;map01100;map01110;m...,Quorum sensing;Plant hormone signal transducti...,72.222222


### 3. Uniting pathways information in BGCs table

In [50]:
df_bgc = pd.read_csv("/home/pedro/script_busca_uniprot/resultado_alinhamento/bgc_table_with_alignment_information.csv")
df_bgc

,Unnamed: 0,BGC_ID,Reference_BGC,distance,jaccard,adjacency,dss,reference_BGC_class,reference_compound_name,has_resistance_protein,...,resistance_protein_count,antismash_annotation,num_proteins_antismash_annotation,strand,coordinates,UniProt_alignment_hits,UniProt_protein_descriptions,UniProt_protein_organism,UniProt_protein_TaxID,UniProt_protein_gene
0,0,MGYG000296006_405,BGC0002398,0.47,1.00,1.00,0.38,terpene,(-)-ent-quiannulatene,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,MGYG000296006_71,BGC0002132,0.74,0.67,0.50,0.14,ribosomal,group 1 methanobactin,False,...,NaN,NaN,NaN,+,ctg71_3 --> ctg71_4 --> ctg71_5 --> ctg71_6 --...,NaN,NaN,NaN,NaN,NaN
2,2,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,...,NaN,NaN,NaN,+,ctg2_6 --> ctg2_7 --> ctg2_8 --> ctg2_9 --> ct...,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP
3,3,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,...,NaN,NaN,NaN,-,ctg2_2 <-- ctg2_3 <-- ctg2_4 <-- ctg2_5 / ctg2...,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP
4,4,MGYG000296008_22,BGC0000924,0.81,0.50,0.33,0.10,other,pyrrolnitrin,False,...,NaN,NaN,NaN,-,ctg22_2 <-- ctg22_3 <-- ctg22_4 <-- ctg22_5 <-...,"MGYG000296008_ctg22_13, MGYG000296008_ctg22_15...",Amino acid adenylation domain-containing prote...,"Paenibacillus apii, Streptomyces violaceoruber...","1850370, 1935, 2571141, 294","G5B47_10895, B1H20_06975, FCI23_38020, prnD"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,165,MGYG000296076_1,BGC0000656,0.91,0.33,0.00,0.04,terpene,zeaxanthin,False,...,NaN,NaN,NaN,-,ctg1_139 <-- ctg1_140 <-- ctg1_141 / ctg1_148 ...,"MGYG000296076_ctg1_130, MGYG000296076_ctg1_133...",Aminotransferase class I/II-fold pyridoxal pho...,"Amycolatopsis pithecellobii, Streptomyces nive...","664692, 193462, 389348","GKO32_27080, novJ, PNK_1118"
166,166,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,...,NaN,carotenoid,3.0,+,3/548 --> ctg10_23 --> ctg10_24 --> ctg10_25,"MGYG000296076_ctg10_30, MGYG000296076_ctg10_37","SDR family oxidoreductase, Lysine--tRNA ligase","Streptomyces rubrogriseus, Capillimicrobium pa...","194673, 2884022","G3I66_28110, epmA"
167,167,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,...,NaN,carotenoid,3.0,-,ctg10_29 <-- ctg10_30 <-- ctg10_31 <-- ctg10_3...,"MGYG000296076_ctg10_30, MGYG000296076_ctg10_37","SDR family oxidoreductase, Lysine--tRNA ligase","Streptomyces rubrogriseus, Capillimicrobium pa...","194673, 2884022","G3I66_28110, epmA"
168,168,MGYG000296076_4,BGC0002865,0.56,1.00,1.00,0.26,PKS,Aloesone,False,...,NaN,NaN,NaN,+,ctg4_63 --> ctg4_64 --> ctg4_65 --> ctg4_66 / ...,"MGYG000296076_ctg4_64, MGYG000296076_ctg4_71","Lipid A 4'-phosphatase, ABC transporter family...",Porphyromonas gingivalis (strain ATCC 33277 / ...,"431947, 46176","lpxF, EDD27_3505"


In [52]:
df_bgc = pd.merge(df_bgc, df_groupby, on="BGC_ID", how="left")
df_bgc

,Unnamed: 0,BGC_ID,Reference_BGC,distance,jaccard,adjacency,dss,reference_BGC_class,reference_compound_name,has_resistance_protein,...,strand,coordinates,UniProt_alignment_hits,UniProt_protein_descriptions,UniProt_protein_organism,UniProt_protein_TaxID,UniProt_protein_gene,pathway,description,percentual
0,0,MGYG000296006_405,BGC0002398,0.47,1.00,1.00,0.38,terpene,(-)-ent-quiannulatene,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,map00790;map01100;map01240;map00900;map01110,Folate biosynthesis;Metabolic pathways;Biosynt...,50.000000
1,1,MGYG000296006_71,BGC0002132,0.74,0.67,0.50,0.14,ribosomal,group 1 methanobactin,False,...,+,ctg71_3 --> ctg71_4 --> ctg71_5 --> ctg71_6 --...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,...,+,ctg2_6 --> ctg2_7 --> ctg2_8 --> ctg2_9 --> ct...,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP,map00400;map01100;map01110;map01230;map02024;m...,"Phenylalanine, tyrosine and tryptophan biosynt...",76.470588
3,3,MGYG000296008_2,BGC0002283,0.71,0.50,0.40,0.23,terpene,sodorifen,False,...,-,ctg2_2 <-- ctg2_3 <-- ctg2_4 <-- ctg2_5 / ctg2...,MGYG000296008_ctg2_15,Undecaprenyl-diphosphatase,Sedimentisphaera cyanobacteriorum,1940790,uppP,map00400;map01100;map01110;map01230;map02024;m...,"Phenylalanine, tyrosine and tryptophan biosynt...",76.470588
4,4,MGYG000296008_22,BGC0000924,0.81,0.50,0.33,0.10,other,pyrrolnitrin,False,...,-,ctg22_2 <-- ctg22_3 <-- ctg22_4 <-- ctg22_5 <-...,"MGYG000296008_ctg22_13, MGYG000296008_ctg22_15...",Amino acid adenylation domain-containing prote...,"Paenibacillus apii, Streptomyces violaceoruber...","1850370, 1935, 2571141, 294","G5B47_10895, B1H20_06975, FCI23_38020, prnD",map00984;map01100;map01120;map00360;map00362;m...,Steroid degradation;Metabolic pathways;Microbi...,33.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
165,165,MGYG000296076_1,BGC0000656,0.91,0.33,0.00,0.04,terpene,zeaxanthin,False,...,-,ctg1_139 <-- ctg1_140 <-- ctg1_141 / ctg1_148 ...,"MGYG000296076_ctg1_130, MGYG000296076_ctg1_133...",Aminotransferase class I/II-fold pyridoxal pho...,"Amycolatopsis pithecellobii, Streptomyces nive...","664692, 193462, 389348","GKO32_27080, novJ, PNK_1118",map00260;map01100;map01110;map01120;map01230;m...,"Glycine, serine and threonine metabolism;Metab...",71.428571
166,166,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,...,+,3/548 --> ctg10_23 --> ctg10_24 --> ctg10_25,"MGYG000296076_ctg10_30, MGYG000296076_ctg10_37","SDR family oxidoreductase, Lysine--tRNA ligase","Streptomyces rubrogriseus, Capillimicrobium pa...","194673, 2884022","G3I66_28110, epmA",map02024;map04075;map00906;map01100;map01110;m...,Quorum sensing;Plant hormone signal transducti...,72.222222
167,167,MGYG000296076_10,BGC0000647,0.60,1.00,1.00,0.19,terpene,carotenoid,False,...,-,ctg10_29 <-- ctg10_30 <-- ctg10_31 <-- ctg10_3...,"MGYG000296076_ctg10_30, MGYG000296076_ctg10_37","SDR family oxidoreductase, Lysine--tRNA ligase","Streptomyces rubrogriseus, Capillimicrobium pa...","194673, 2884022","G3I66_28110, epmA",map02024;map04075;map00906;map01100;map01110;m...,Quorum sensing;Plant hormone signal transducti...,72.222222
168,168,MGYG000296076_4,BGC0002865,0.56,1.00,1.00,0.26,PKS,Aloesone,False,...,+,ctg4_63 --> ctg4_64 --> ctg4_65 --> ctg4_66 / ...,"MGYG000296076_ctg4_64, MGYG000296076_ctg4_71","Lipid A 4'-phosphatase, ABC transporter family...",Porphyromonas gingivalis (strain ATCC 33277 / ...,"431947, 46176","lpxF, EDD27_3505",map00550;map00552;map00130;map01100;map01110;m...,Peptidoglycan biosynthesis;Teichoic acid biosy...,54.838710
